# FIT3182 Assignment 2 — Kafka Producer C

Standalone producer for `camera_event_C.csv` → Kafka topic `camera-events-C`.

**Run this in a separate terminal/notebook** while the main streaming notebook is consuming.
Ensure Kafka is running on `localhost:9092`.


In [ ]:
import json, csv, time
from kafka import KafkaProducer

# Initialising parameters for Kafka Producer
ip = "localhost:9092"
file_p = "../data/camera_event_C.csv"
topic = "camera-events-C"
prod_id = "C"
inter = 2

In [ ]:
producer = KafkaProducer(
    bootstrap_servers = ip,
    value_serializer = lambda val : json.dumps(val).encode('utf-8'),
    key_serializer = lambda key : key.encode('utf-8') if key else None,
    linger_ms = 50,
    acks = 'all', # all for leader-plus-follower acknoledgement of write
    retries = 3
)

batches = {}

# Extract data from CSV file
with open(file_p, 'r') as file:
    for row in csv.DictReader(file):
        batch_id = int(row['batch_id'])
        batches.setdefault(batch_id, []).append(row)

# Get sorted list of batch IDs
sorted_ids = sorted(batches.keys())

In [ ]:
# Publish batch events
for batch_id in sorted_ids:
    for row in batches[batch_id]:
        event = {
            "event_id": row["event_id"],
            "batch_id": int(row["batch_id"]),
            "car_plate": row["car_plate"].strip(),
            "camera_id": int(row["camera_id"]),
            "timestamp": row["timestamp"].strip(),
            "speed_reading": float(row["speed_reading"]),
            "producer_id": prod_id
        }

        producer.send(topic, key = event["car_plate"], value = event)
        producer.flush()

    time.sleep(inter)

producer.flush()
producer.close()